In [1]:
import os
import pandas as pd
import numpy as np
import cv2
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

I0000 00:00:1785558826.837619  239405 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1785558827.002815  239405 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785558830.387174  239405 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [4]:
train_data = pd.read_csv("../dataset/train_metadata.csv")
val_data = pd.read_csv("../dataset/val_metadata.csv")
test_data = pd.read_csv("../dataset/test_metadata.csv")

In [5]:
IMAGE_SIZE = 224
from tensorflow.keras.applications.resnet50 import preprocess_input


def preprocess_image(image_path):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
    image = cv2.resize(image,(IMAGE_SIZE, IMAGE_SIZE))
    image = preprocess_input(image)
    return image.astype(np.float32)

In [6]:
sample_image = preprocess_image(
    train_data.iloc[0]["image_path"]
)


sample_image.shape

(224, 224, 3)

In [7]:
def create_dataset(data):
    image_paths = data["image_path"].values
    labels = data["label"].values
    dataset = tf.data.Dataset.from_tensor_slices((image_paths,labels))
    def load_image(path,label):
        image = tf.numpy_function(preprocess_image,[path],tf.float32)
        image.set_shape((224,224,3))
        return image,label
    dataset = dataset.map(load_image,num_parallel_calls=tf.data.AUTOTUNE)
    return dataset

In [8]:
train_dataset = create_dataset(train_data)
val_dataset = create_dataset(val_data)
test_dataset = create_dataset(test_data)

E0000 00:00:1785558863.504926  239405 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [9]:
data_augmentation = tf.keras.Sequential([

    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1)

])

In [10]:
def apply_augmentation(image,label):
    image = data_augmentation(image)
    return image,label
train_dataset = train_dataset.map(apply_augmentation,num_parallel_calls=tf.data.AUTOTUNE)

In [11]:
BATCH_SIZE = 32
train_dataset = (train_dataset.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
val_dataset = (val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
test_dataset = (test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

In [12]:
class_weights = compute_class_weight(class_weight="balanced",classes=np.unique(train_data["label"]),y=train_data["label"])
class_weights

array([ 4.37305053,  2.78174603,  1.30224782, 12.3633157 ,  1.2855309 ,
        0.21338772, 10.11544012])

In [13]:
class_weights = dict(enumerate(class_weights))
class_weights

{0: np.float64(4.37305053025577),
 1: np.float64(2.7817460317460316),
 2: np.float64(1.3022478172023035),
 3: np.float64(12.36331569664903),
 4: np.float64(1.285530900421786),
 5: np.float64(0.21338772031292808),
 6: np.float64(10.115440115440116)}

In [28]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models


base_model = ResNet50(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_model.trainable = False

resnet50_model = models.Sequential([

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation="softmax")

])


resnet50_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,850,887 (90.98 MB)

 Trainable params: 263,175 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [29]:
resnet50_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

In [30]:
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)

In [31]:
history_eff = resnet50_model.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5


/Users/aximsoft/Documents/untitled folder/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 444s 2s/step - accuracy: 0.2669 - loss: 1.9263 - val_accuracy: 0.3016 - val_loss: 1.7794
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 451s 2s/step - accuracy: 0.3963 - loss: 1.5467 - val_accuracy: 0.3768 - val_loss: 1.6129
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 463s 2s/step - accuracy: 0.4595 - loss: 1.4337 - val_accuracy: 0.3808 - val_loss: 1.6272
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 430s 2s/step - accuracy: 0.4762 - loss: 1.3662 - val_accuracy: 0.3842 - val_loss: 1.5999
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 433s 2s/step - accuracy: 0.4937 - loss: 1.2923 - val_accuracy: 0.4101 - val_loss: 1.5826
Restoring model weights from the end of the best epoch: 5.


In [32]:
train_loss, train_accuracy = resnet50_model.evaluate(train_dataset)
val_loss, val_accuracy = resnet50_model.evaluate(val_dataset)
test_loss, test_accuracy = resnet50_model.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 370s 2s/step - accuracy: 0.5805 - loss: 1.1790
47/47 ━━━━━━━━━━━━━━━━━━━━ 77s 2s/step - accuracy: 0.4101 - loss: 1.5826
47/47 ━━━━━━━━━━━━━━━━━━━━ 76s 2s/step - accuracy: 0.3886 - loss: 1.6219


In [33]:
resnet50_model.save("models/resnet50_model.keras")

In [34]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models


base_model = ResNet50(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_model.trainable = False

resnet50_model_sgd= models.Sequential([

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation="softmax")

])


resnet50_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,377,239 (92.99 MB)

 Trainable params: 263,175 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

 Optimizer params: 526,352 (2.01 MB)

In [35]:
resnet50_model_sgd.compile(

    optimizer=tf.keras.optimizers.SGD(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_res_sgd = resnet50_model_sgd.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 407s 2s/step - accuracy: 0.1969 - loss: 2.4170 - val_accuracy: 0.1917 - val_loss: 2.0541
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 223s 1s/step - accuracy: 0.2110 - loss: 2.1670 - val_accuracy: 0.1951 - val_loss: 2.0347
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 308s 1s/step - accuracy: 0.2284 - loss: 2.0150 - val_accuracy: 0.2124 - val_loss: 1.9961
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 391s 2s/step - accuracy: 0.2609 - loss: 1.9624 - val_accuracy: 0.2150 - val_loss: 1.9956
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 407s 2s/step - accuracy: 0.2719 - loss: 1.8925 - val_accuracy: 0.2270 - val_loss: 1.9563
Restoring model weights from the end of the best epoch: 5.


In [36]:
train_loss, train_accuracy = resnet50_model_sgd.evaluate(train_dataset)
val_loss, val_accuracy = resnet50_model_sgd.evaluate(val_dataset)
test_loss, test_accuracy = resnet50_model_sgd.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 329s 1s/step - accuracy: 0.4088 - loss: 1.6685
47/47 ━━━━━━━━━━━━━━━━━━━━ 69s 1s/step - accuracy: 0.2270 - loss: 1.9563
47/47 ━━━━━━━━━━━━━━━━━━━━ 68s 1s/step - accuracy: 0.2029 - loss: 1.9986


In [37]:
resnet50_model_sgd.save("models/resnet50_model_sgd.keras")

In [38]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models


base_model = ResNet50(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_model.trainable = False

resnet50_model_RMSprop = models.Sequential([

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation="softmax")

])


resnet50_model_RMSprop.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,850,887 (90.98 MB)

 Trainable params: 263,175 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [39]:
resnet50_model_RMSprop.compile(

    optimizer=tf.keras.optimizers.RMSprop(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_res_RMSprop = resnet50_model_RMSprop.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 405s 2s/step - accuracy: 0.3532 - loss: 2.0194 - val_accuracy: 0.4867 - val_loss: 1.5018
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 406s 2s/step - accuracy: 0.4740 - loss: 1.6361 - val_accuracy: 0.4927 - val_loss: 1.4343
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 401s 2s/step - accuracy: 0.5201 - loss: 1.5169 - val_accuracy: 0.5027 - val_loss: 1.4002
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 415s 2s/step - accuracy: 0.5518 - loss: 1.4048 - val_accuracy: 0.5126 - val_loss: 1.3585
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 392s 2s/step - accuracy: 0.5705 - loss: 1.3549 - val_accuracy: 0.5439 - val_loss: 1.2873
Restoring model weights from the end of the best epoch: 5.


In [40]:
train_loss, train_accuracy = resnet50_model_RMSprop.evaluate(train_dataset)
val_loss, val_accuracy = resnet50_model_RMSprop.evaluate(val_dataset)
test_loss, test_accuracy = resnet50_model_RMSprop.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 330s 1s/step - accuracy: 0.6512 - loss: 1.0057
47/47 ━━━━━━━━━━━━━━━━━━━━ 86s 2s/step - accuracy: 0.5439 - loss: 1.2873
47/47 ━━━━━━━━━━━━━━━━━━━━ 86s 2s/step - accuracy: 0.5136 - loss: 1.3621


In [41]:
resnet50_model_RMSprop.save("models/resnet50_model_RMSprop.keras")

In [42]:
train_dataset = create_dataset(train_data)
val_dataset = create_dataset(val_data)
test_dataset = create_dataset(test_data)
 
train_dataset = train_dataset.map(
    apply_augmentation,
    num_parallel_calls=tf.data.AUTOTUNE
)
 
BATCH_SIZE = 64
 
train_dataset = (
    train_dataset
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
 
val_dataset = (
    val_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
 
test_dataset = (
    test_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [43]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models


base_model = ResNet50(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_model.trainable = False

resnet50_model_64= models.Sequential([

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation="softmax")

])


resnet50_model_64.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,850,887 (90.98 MB)

 Trainable params: 263,175 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [44]:
resnet50_model_64.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_res_64= resnet50_model_64.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 335s 3s/step - accuracy: 0.2900 - loss: 2.1823 - val_accuracy: 0.3236 - val_loss: 1.7241
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 278s 2s/step - accuracy: 0.3795 - loss: 1.6924 - val_accuracy: 0.2776 - val_loss: 1.7737
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 236s 2s/step - accuracy: 0.4334 - loss: 1.5457 - val_accuracy: 0.3808 - val_loss: 1.6219
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 227s 2s/step - accuracy: 0.4736 - loss: 1.4379 - val_accuracy: 0.4048 - val_loss: 1.5399
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 224s 2s/step - accuracy: 0.5050 - loss: 1.3596 - val_accuracy: 0.4361 - val_loss: 1.4457
Restoring model weights from the end of the best epoch: 5.


In [45]:
train_loss, train_accuracy = resnet50_model_64.evaluate(train_dataset)
val_loss, val_accuracy = resnet50_model_64.evaluate(val_dataset)
test_loss, test_accuracy = resnet50_model_64.evaluate(test_dataset)

110/110 ━━━━━━━━━━━━━━━━━━━━ 188s 2s/step - accuracy: 0.5869 - loss: 1.1382
24/24 ━━━━━━━━━━━━━━━━━━━━ 39s 2s/step - accuracy: 0.4361 - loss: 1.4457
24/24 ━━━━━━━━━━━━━━━━━━━━ 39s 2s/step - accuracy: 0.4192 - loss: 1.4884


In [46]:
resnet50_model_64.save("models/resnet50_model_64.keras")

In [47]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

# Load MobileNetV2

base_model = ResNet50(

    weights="imagenet",

    include_top=False,

    input_shape=(224,224,3)

)

# Fine-Tuning

base_model.trainable = True

# Freeze all layers except the last 30

for layer in base_model.layers[:-30]:

    layer.trainable = False

# Build Model

res_ft = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(
        128,
        activation="relu"
    ),

    layers.Dropout(0.5),

    layers.Dense(
        7,
        activation="softmax"
    )

])

res_ft.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_5      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,850,887 (90.98 MB)

 Trainable params: 14,713,351 (56.13 MB)

 Non-trainable params: 9,137,536 (34.86 MB)

In [48]:
res_ft.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_res_ft = res_ft.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 312s 3s/step - accuracy: 0.4302 - loss: 1.5445 - val_accuracy: 0.6398 - val_loss: 1.0709
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 305s 3s/step - accuracy: 0.6093 - loss: 1.0014 - val_accuracy: 0.5300 - val_loss: 1.2600
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 951s 9s/step - accuracy: 0.6408 - loss: 0.8293 - val_accuracy: 0.6398 - val_loss: 1.1016
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 309s 3s/step - accuracy: 0.6729 - loss: 0.6902 - val_accuracy: 0.5413 - val_loss: 1.4999
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 1.


In [49]:
train_loss, train_accuracy = res_ft.evaluate(train_dataset)
val_loss, val_accuracy = res_ft.evaluate(val_dataset)
test_loss, test_accuracy = res_ft.evaluate(test_dataset)

110/110 ━━━━━━━━━━━━━━━━━━━━ 186s 2s/step - accuracy: 0.7158 - loss: 0.7594
24/24 ━━━━━━━━━━━━━━━━━━━━ 39s 2s/step - accuracy: 0.6398 - loss: 1.0709
24/24 ━━━━━━━━━━━━━━━━━━━━ 39s 2s/step - accuracy: 0.6114 - loss: 1.1587


In [50]:
res_ft.save("models/res_ft.keras")

In [51]:
import keras_tuner as kt
import tensorflow as tf

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam, RMSprop

num_classes = 7

def build_model(hp):

    base_model = ResNet50(

        weights="imagenet",

        include_top=False,

        input_shape=(224,224,3)

    )

    base_model.trainable = False

    res_model = models.Sequential([

        base_model,

        layers.GlobalAveragePooling2D(),

        layers.Dense(

            units=hp.Choice(

                "dense_units",

                [128,256,512]

            ),

            activation="relu"

        ),

        layers.Dropout(

            hp.Choice(

                "dropout",

                [0.3,0.5,0.6]

            )

        ),

        layers.Dense(

            num_classes,

            activation="softmax"

        )

    ])

    learning_rate = hp.Choice(

        "learning_rate",

        [1e-3,1e-4,1e-5]

    )

    optimizer = hp.Choice(

        "optimizer",

        ["adam","rmsprop"]

    )

    if optimizer == "adam":

        opt = Adam(

            learning_rate=learning_rate

        )

    else:

        opt = RMSprop(

            learning_rate=learning_rate

        )

    res_model.compile(

        optimizer=opt,

        loss="sparse_categorical_crossentropy",

        metrics=["accuracy"]

    )

    return res_model

In [53]:
tuner = kt.RandomSearch(

    build_model,

    objective="val_accuracy",

    max_trials=3,

    directory="resnet_tuner",

    project_name="mobilenet_hyperparameter"

)

In [54]:
tuner.search(

    train_dataset,

    validation_data=val_dataset,

    epochs=5,

    class_weight=class_weights

)

Trial 3 Complete [00h 19m 28s]
val_accuracy: 0.595872163772583

Best val_accuracy So Far: 0.6138482093811035
Total elapsed time: 01h 01m 33s


In [55]:
best_eff_net = tuner.get_best_models(1)[0]

/Users/aximsoft/Documents/untitled folder/.venv/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(store)


In [56]:
best_hps = tuner.get_best_hyperparameters(
    num_trials=1
)[0]
print(best_hps.values)

{'dense_units': 512, 'dropout': 0.5, 'learning_rate': 0.001, 'optimizer': 'rmsprop'}


In [14]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models


base_model = ResNet50(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_model.trainable = False

resnet50_model_final = models.Sequential([

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512,activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation="softmax")

])


resnet50_model_final.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     1,049,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         3,591 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,640,391 (94.00 MB)

 Trainable params: 1,052,679 (4.02 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [15]:
resnet50_model_final.compile(

    optimizer=tf.keras.optimizers.RMSprop(
        learning_rate=0.001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_res_final = resnet50_model_final.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5


/home/aximsoft/snap/code/253/.local/share/virtualenvs/SkinCancer_Disease-Fur3-H-s/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
W0000 00:00:1785558893.209273  255926 cpu_allocator_impl.cc:82] Allocation of 102760448 exceeds 10% of free system memory.
W0000 00:00:1785558893.464142  255926 cpu_allocator_impl.cc:82] Allocation of 106463232 exceeds 10% of free system memory.
W0000 00:00:1785558893.591443  255929 cpu_allocator_impl.cc:82] Allocation of 102760448 exceeds 10% of free system memory.
W0000 00:00:1785558893.850107  255926 cpu_allocator_impl.cc:82] Allocation of 102760448 exceeds 10% of free system memory.
W0000 00:00:1785558894.099384  255926 cpu_allocator_impl.cc:82] Allocation of 102760448 exceeds 10% of free sy

220/220 ━━━━━━━━━━━━━━━━━━━━ 981s 4s/step - accuracy: 0.4508 - loss: 2.1715 - val_accuracy: 0.4541 - val_loss: 1.4560
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 667s 3s/step - accuracy: 0.5417 - loss: 1.6106 - val_accuracy: 0.4654 - val_loss: 1.6732
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 549s 2s/step - accuracy: 0.5563 - loss: 1.5096 - val_accuracy: 0.5546 - val_loss: 1.2907
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 603s 3s/step - accuracy: 0.5783 - loss: 1.4140 - val_accuracy: 0.5240 - val_loss: 1.3765
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 665s 3s/step - accuracy: 0.5934 - loss: 1.4515 - val_accuracy: 0.6644 - val_loss: 0.9303
Restoring model weights from the end of the best epoch: 5.


In [16]:
train_loss, train_accuracy = resnet50_model_final.evaluate(train_dataset)
val_loss, val_accuracy = resnet50_model_final.evaluate(val_dataset)
test_loss, test_accuracy = resnet50_model_final.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 514s 2s/step - accuracy: 0.7173 - loss: 0.7462
47/47 ━━━━━━━━━━━━━━━━━━━━ 94s 2s/step - accuracy: 0.6644 - loss: 0.9303
47/47 ━━━━━━━━━━━━━━━━━━━━ 95s 2s/step - accuracy: 0.6454 - loss: 1.0330


In [18]:
resnet50_model_final.save("../models/resnet50_model_final.keras")